# Lab 03 — Mapping use cases and selecting models

**Level:** Intermediate · **Duration:** ~45 minutes · **Covers:** H1 enterprise use-case mapping (30 min) · H5 model selection exercise (15 min)

Lab guide: [`../lab-03-use-cases-model-selection.md`](../lab-03-use-cases-model-selection.md) — read each part there first; this notebook holds the code.

A team exercise with code. You map four IT scenarios to an agent type and a model family, ask both providers to review your mapping, build a cost model, and turn it into a weighted decision that fills the model comparison sheet. Four functions are yours (marked `TODO`); check cells test their contracts.

In [ ]:
# (notebook) Setup: run this cell first. It loads the repo-root .env and creates both SDK clients.
import json
import os
import sys
import time
from pathlib import Path

import anthropic
import openai
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env

# labs/module-01 holds data/ and templates/; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-01" and (p / "data").is_dir())
DATA_DIR = LAB_DIR / "data"
SETUP_DIR = LAB_DIR.parent / "setup"

# Model IDs come from .env, never from code (labs/setup/.env.example has the defaults).
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "").strip()
OPENAI_SMALL_MODEL = os.getenv("OPENAI_SMALL_MODEL", "").strip()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "").strip()
ANTHROPIC_SMALL_MODEL = os.getenv("ANTHROPIC_SMALL_MODEL", "").strip()

# Each SDK reads its key from the environment; a client exists only when key and model are set.
openai_client = openai.OpenAI() if os.getenv("OPENAI_API_KEY") and OPENAI_MODEL else None
claude_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") and ANTHROPIC_MODEL else None


def available(provider: str) -> bool:
    """True if `provider` is configured; otherwise print why the cell skips it."""
    client = {"openai": openai_client, "anthropic": claude_client}[provider]
    if client is None:
        print(f"⚠️  {provider}: not configured, skipping. Add its key and model to .env, then restart the kernel.")
    return client is not None


def active_providers() -> list[str]:
    """The configured providers, with a printed note for each one that is skipped."""
    return [p for p in ("openai", "anthropic") if available(p)]


def claude_text(message) -> str:
    """Join the text blocks of a Claude response; thinking blocks are skipped."""
    return "".join(block.text for block in message.content if block.type == "text")


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(f"Lab folder : {LAB_DIR}")
print(f"OpenAI     : {'✅ ' + OPENAI_MODEL if openai_client else '❌ OPENAI_API_KEY or OPENAI_MODEL missing'}")
print(f"Claude     : {'✅ ' + ANTHROPIC_MODEL if claude_client else '❌ ANTHROPIC_API_KEY or ANTHROPIC_MODEL missing'}")
if not (openai_client or claude_client):
    print(
        "\n❌ No provider is configured. Copy labs/setup/.env.example to the repo-root .env, add your keys,\n"
        "   restart the kernel and run this cell again. Until then, cells that call a model are skipped."
    )

from typing import Literal

from pydantic import BaseModel, Field, ValidationError

SCENARIOS = {s["id"]: s for s in json.loads((DATA_DIR / "use_case_scenarios.json").read_text(encoding="utf-8"))}
CATALOGUE_DOC = json.loads((DATA_DIR / "model_catalogue.json").read_text(encoding="utf-8"))
CATALOGUE = {m["id"]: m for m in CATALOGUE_DOC["models"]}
OUTPUT_DIR = LAB_DIR / "outputs"
print(f"{len(SCENARIOS)} scenarios · {len(CATALOGUE)} models · prices as of {CATALOGUE_DOC['prices_as_of']}")

## Part 1 — Frame the four scenarios (5 min)

Read the four scenarios as a team. For each one, note the volume, the token shape (input and
output per request), the latency target, and the constraints that rule options out: residency,
sensitivity, languages and whether it needs tools or documents.

In [ ]:
# (notebook) The four enterprise IT scenarios.
pd.DataFrame(
    [
        {
            "id": s["id"],
            "title": s["title"],
            "team": s["team"],
            "requests/day": s["requests_per_day"],
            "avg in/out tokens": f"{s['avg_input_tokens']:,} / {s['avg_output_tokens']:,}",
            "max input": f"{s['max_input_tokens']:,}",
            "latency SLO (s)": s["latency_slo_seconds"],
            "tools": s["needs_tools"],
            "documents": s["needs_documents"],
            "batch OK": s["batch_eligible"],
            "residency": s["residency"],
        }
        for s in SCENARIOS.values()
    ]
).set_index("id")

In [ ]:
# (notebook) The agent types you choose from (Module 2 builds the first four; Module 4 the last).
AGENT_TYPES = {
    "single-call classifier": "One request, one structured answer: routing, triage, tagging. No loop, no tools.",
    "extraction pipeline": "A fixed workflow (OCR → LLM extraction → validation) your code orchestrates. Batchable.",
    "RAG assistant": "Retrieve relevant passages, answer from them, cite them. Conversational.",
    "tool-using agent": "The model decides which tools to call in a loop (ReAct); a human approves risky actions.",
    "multi-agent orchestration": "A supervisor routes work to specialist agents; for multi-domain, multi-step work.",
}
display(pd.Series(AGENT_TYPES, name="what it is").to_frame())

pd.DataFrame(CATALOGUE.values())[
    ["id", "provider", "family", "tier", "deployment", "input", "output", "context_window", "vision"]
].set_index("id")

## Part 2 — Map each scenario to an agent type and a model family (25 min)

Work as a team (about 20 minutes of discussion). For every scenario, decide the agent type, the
model tier and the deployment, and name **at least one OpenAI and one Anthropic candidate**. Write
the rationale as you would for an architecture review. The schema below rejects incomplete
mappings; the check cell then tests them against the catalogue.

In [ ]:
# (notebook) The shape of a mapping.
AgentType = Literal[
    "single-call classifier", "extraction pipeline", "RAG assistant", "tool-using agent", "multi-agent orchestration"
]


class UseCaseMapping(BaseModel):
    scenario_id: str
    agent_type: AgentType
    model_tier: Literal["frontier", "balanced", "fast"]
    deployment: Literal["public API", "regional API", "self-hosted open weights"]
    candidates: list[str] = Field(min_length=2, description="model IDs from the catalogue, best first")
    rationale: str = Field(min_length=40)
    risks: list[str] = Field(min_length=1)

In [ ]:
# (notebook) Your team's mappings.
def team_mappings() -> list[UseCaseMapping]:
    """One UseCaseMapping per scenario, UC-1 to UC-4."""
    worked_example = UseCaseMapping(
        scenario_id="UC-1",
        agent_type="single-call classifier",
        model_tier="fast",
        deployment="regional API",
        candidates=["claude-haiku-5-5", "gpt-6-luna"],
        rationale="High volume, tiny outputs and a 2-second SLO: a fast tier with a structured-output schema "
        "is enough, and per-request cost dominates. No tools or loop needed.",
        risks=["misrouted P1s: escalate low-confidence labels to a human", "quality may differ by language"],
    )
    # TODO: add UseCaseMapping objects for UC-2, UC-3 and UC-4 from your team's discussion,
    #       then return all four, e.g. return [worked_example, uc2, uc3, uc4]
    raise NotImplementedError("team_mappings: UC-2, UC-3 and UC-4")

In [ ]:
# (notebook) Check: the mappings are complete and consistent with the catalogue.
def validate_mappings(mappings: list[UseCaseMapping]) -> None:
    assert sorted(m.scenario_id for m in mappings) == sorted(SCENARIOS), "map every scenario exactly once"
    for m in mappings:
        scenario = SCENARIOS[m.scenario_id]
        unknown = [c for c in m.candidates if c not in CATALOGUE]
        assert not unknown, f"{m.scenario_id}: {unknown} not in data/model_catalogue.json"
        providers = {CATALOGUE[c]["provider"] for c in m.candidates if not CATALOGUE[c]["open_weight"]}
        assert {"openai", "anthropic"} <= providers, f"{m.scenario_id}: name an OpenAI AND an Anthropic API model"
        for c in m.candidates:
            window = CATALOGUE[c]["max_input_tokens"] or CATALOGUE[c]["context_window"]
            assert window >= scenario["max_input_tokens"], (
                f"{m.scenario_id}: {c} holds {window:,} tokens but the scenario needs {scenario['max_input_tokens']:,}"
            )
        if m.deployment == "self-hosted open weights":
            assert any(CATALOGUE[c]["open_weight"] for c in m.candidates), f"{m.scenario_id}: self-hosted needs an open-weight candidate"


check("Part 2 — mappings", lambda: validate_mappings(team_mappings()))

In [ ]:
# (notebook) A second opinion from both providers: same prompt, same review schema.
class MappingReview(BaseModel):
    agrees: bool = Field(description="True if the mapping is sound for production")
    biggest_risk: str = Field(description="The single most important risk the team should address")
    overlooked: list[str] = Field(description="Constraints in the scenario the rationale ignores (may be empty)")
    suggested_agent_type: AgentType | None = Field(description="A better agent type, or null if the team's is right")


REVIEW_SYSTEM = (
    "You are a principal architect reviewing an AI use-case mapping before an architecture board. "
    "Judge it against the scenario's volume, latency, residency, sensitivity and accuracy bar. Be concise."
)


def review_prompt(mapping: UseCaseMapping) -> str:
    catalogue = {c: {k: CATALOGUE[c][k] for k in ("tier", "deployment", "input", "output", "context_window")} for c in mapping.candidates}
    return (
        f"<scenario>\n{json.dumps(SCENARIOS[mapping.scenario_id], indent=1)}\n</scenario>\n"
        f"<mapping>\n{mapping.model_dump_json(indent=1)}\n</mapping>\n"
        f"<candidate_models>\n{json.dumps(catalogue, indent=1)}\n</candidate_models>"
    )


def review_openai(mapping: UseCaseMapping) -> MappingReview:
    response = openai_client.responses.parse(
        model=OPENAI_MODEL, instructions=REVIEW_SYSTEM, input=review_prompt(mapping),
        text_format=MappingReview, reasoning={"effort": "low"}, max_output_tokens=3_000,
    )
    return response.output_parsed


def review_claude(mapping: UseCaseMapping) -> MappingReview:
    message = claude_client.messages.parse(
        model=ANTHROPIC_MODEL, max_tokens=3_000, system=REVIEW_SYSTEM, output_config={"effort": "low"},
        messages=[{"role": "user", "content": review_prompt(mapping)}], output_format=MappingReview,
    )
    return message.parsed_output


REVIEW = {"openai": review_openai, "anthropic": review_claude}
reviews = []
try:
    mappings = team_mappings()
except NotImplementedError:
    mappings = []
    print("⏳ Finish team_mappings() first; the reviews run on your four mappings.")
for mapping in mappings:
    for provider in active_providers():
        review = REVIEW[provider](mapping)
        reviews.append({"scenario": mapping.scenario_id, "reviewer": provider, **review.model_dump()})
pd.DataFrame(reviews)

## Part 3 — Estimate tokens and cost (7 min)

Measure before you model: count the tokens of each scenario's sample request with both providers'
counters, and time one real request on the main and small tier. Then write the cost function.

In [ ]:
# (notebook) Tokens in each scenario's sample request (system + user), counted by each provider.
import tiktoken

try:
    o200k = tiktoken.get_encoding("o200k_base")  # offline approximation, cached by prefetch_models.py
except Exception as exc:
    o200k = None
    print(f"⚠️  o200k_base unavailable ({exc.__class__.__name__}); the offline column is skipped.")


def sample_tokens(scenario: dict) -> dict:
    sample = scenario["sample_request"]
    row = {"scenario": scenario["id"], "planning avg input": scenario["avg_input_tokens"]}
    if o200k is not None:
        row["o200k (offline)"] = len(o200k.encode(sample["system"] + sample["user"]))
    if openai_client:
        row["OpenAI count"] = openai_client.responses.input_tokens.count(
            model=OPENAI_MODEL, instructions=sample["system"], input=sample["user"]
        ).input_tokens
    if claude_client:
        row["Claude count"] = claude_client.messages.count_tokens(
            model=ANTHROPIC_MODEL, system=sample["system"], messages=[{"role": "user", "content": sample["user"]}]
        ).input_tokens
    return row


pd.DataFrame([sample_tokens(s) for s in SCENARIOS.values()]).set_index("scenario")

In [ ]:
# (notebook) Measure, don't guess: the UC-1 sample request on the main and the small tier of each provider.
uc1 = SCENARIOS["UC-1"]["sample_request"]
probe = []
targets = [("openai", OPENAI_MODEL), ("openai", OPENAI_SMALL_MODEL), ("anthropic", ANTHROPIC_MODEL), ("anthropic", ANTHROPIC_SMALL_MODEL)]
for provider, model_id in targets:
    if not model_id or {"openai": openai_client, "anthropic": claude_client}[provider] is None:
        continue
    started = time.perf_counter()
    if provider == "openai":
        r = openai_client.responses.create(
            model=model_id, instructions=uc1["system"], input=uc1["user"], reasoning={"effort": "low"}, max_output_tokens=1_000
        )
        text, tokens_in, tokens_out = r.output_text, r.usage.input_tokens, r.usage.output_tokens
    else:
        m = claude_client.messages.create(
            model=model_id, max_tokens=1_000, system=uc1["system"], output_config={"effort": "low"},
            messages=[{"role": "user", "content": uc1["user"]}],
        )
        text, tokens_in, tokens_out = claude_text(m), m.usage.input_tokens, m.usage.output_tokens
    price = CATALOGUE.get(model_id, {})
    cost = (tokens_in * price["input"] + tokens_out * price["output"]) / 1e6 if price.get("input") is not None else None
    probe.append({"model": model_id, "latency (s)": round(time.perf_counter() - started, 2), "in": tokens_in, "out": tokens_out,
                  "cost per 1,000 requests ($)": round(cost * 1000, 3) if cost is not None else None, "answer": text.strip()[:60]})
if not probe:
    print("⚠️  No provider configured: skipping the tier probe.")
pd.DataFrame(probe)

### The cost function

Contract for `monthly_cost(model, scenario, *, cache_hit_rate=0.0, batch=False, days=30, self_hosted_monthly_usd=None)`:

1. Prices in the catalogue are **USD per 1M tokens**. Per request:
   `input_tokens × input price + output_tokens × output price`, using the scenario's `avg_input_tokens`
   and `avg_output_tokens`.
2. A fraction `cache_hit_rate` of the input tokens is billed at `cached_input` (when the model has one).
   Cache-write premiums are ignored in this exercise.
3. If the model has a `long_context` entry and `avg_input_tokens` exceeds its `threshold_tokens`, use the
   `long_context` prices for the **whole** request.
4. `batch=True` halves the total (both providers' Batch APIs bill at 50%).
5. Monthly = per-request cost × `requests_per_day` × `days`, rounded to cents.
6. A model without token prices (open weights) costs `self_hosted_monthly_usd`; if that is `None`, raise `ValueError`.

In [ ]:
# (notebook) Monthly cost of running one scenario on one model.
def monthly_cost(
    model: dict,
    scenario: dict,
    *,
    cache_hit_rate: float = 0.0,
    batch: bool = False,
    days: int = 30,
    self_hosted_monthly_usd: float | None = None,
) -> float:
    """USD per month for `scenario` on `model`, following the six-point contract above."""
    # TODO: implement the contract. The check cell below has worked numbers to aim for.
    raise NotImplementedError("monthly_cost")

In [ ]:
# (notebook) Check: monthly_cost against hand-calculated cases.
TEST_MODEL = {"id": "test", "input": 1.0, "cached_input": 0.1, "output": 5.0,
              "long_context": {"threshold_tokens": 1_500, "input": 2.0, "cached_input": 0.2, "output": 7.5}}
TEST_SCENARIO = {"requests_per_day": 1_000, "avg_input_tokens": 1_000, "avg_output_tokens": 500}


def check_monthly_cost():
    # 1,000 in × $1 + 500 out × $5 = $0.0035 per request × 1,000 × 30 = $105.00
    assert monthly_cost(TEST_MODEL, TEST_SCENARIO) == 105.00
    # half the input cached: 500 × $1 + 500 × $0.10 + 500 × $5 = $0.00305 → $91.50
    assert monthly_cost(TEST_MODEL, TEST_SCENARIO, cache_hit_rate=0.5) == 91.50
    assert monthly_cost(TEST_MODEL, TEST_SCENARIO, batch=True) == 52.50
    # 2,000 input tokens is over the 1,500 threshold: 2,000 × $2 + 500 × $7.5 = $0.00775 → $232.50
    assert monthly_cost(TEST_MODEL, {**TEST_SCENARIO, "avg_input_tokens": 2_000}) == 232.50
    open_model = {"id": "open", "input": None}
    assert monthly_cost(open_model, TEST_SCENARIO, self_hosted_monthly_usd=4_000) == 4_000.00
    try:
        monthly_cost(open_model, TEST_SCENARIO)
    except ValueError:
        pass
    else:
        raise AssertionError("an open-weight model without self_hosted_monthly_usd must raise ValueError")


check("Part 3 — monthly_cost", check_monthly_cost)

In [ ]:
# (notebook) Every model × every scenario, with caching and batching where the scenario allows them.
# Your infrastructure estimate for self-hosting, in USD/month (GPUs, ops time). None = not estimated yet.
SELF_HOSTED_MONTHLY_USD = {"gpt-oss-120b": None, "gpt-oss-20b": None}


def cost_or_none(model: dict, scenario: dict) -> float | None:
    try:
        return monthly_cost(
            model, scenario,
            cache_hit_rate=scenario["cacheable_prefix_share"],
            batch=scenario["batch_eligible"],
            self_hosted_monthly_usd=SELF_HOSTED_MONTHLY_USD.get(model["id"]),
        )
    except (ValueError, NotImplementedError):
        return None


cost_table = pd.DataFrame(
    {sid: {mid: cost_or_none(m, s) for mid, m in CATALOGUE.items()} for sid, s in SCENARIOS.items()}
)
cost_table.style.format("${:,.0f}", na_rep="—")

## Part 4 — Score, decide and write it up (8 min)

A weighted decision matrix for **one** scenario your team picks. Ratings run from 1 to 5. Five
criteria come from transparent starter rules (argue with them and change them); **cost** comes from
your cost table. Write the two scoring functions, look at the ranking, test how sensitive it is, and
export the filled comparison sheet.

In [ ]:
# (notebook) Pick the scenario, the weights (they must sum to 1.0) and look at the starter ratings.
SCENARIO_ID = "UC-1"  # the guide's §8.4 example; try "UC-3" next
WEIGHTS = {
    "capability": 0.25,
    "latency": 0.10,
    "cost": 0.25,
    "residency": 0.20,
    "multilingual": 0.10,
    "operability": 0.10,
}


def starter_ratings(model: dict, scenario: dict) -> dict[str, float]:
    """Rule-of-thumb ratings (1-5) for every criterion except cost. Change the rules if you disagree."""
    tier = model["tier"]
    hard = scenario["needs_tools"] or scenario["avg_input_tokens"] > 20_000
    capability = {"frontier": 5, "balanced": 4, "fast": 3 if hard else 4}[tier]
    latency = {"fast": 5, "balanced": 4, "frontier": 3}[tier]
    if scenario["latency_slo_seconds"] >= 600:
        latency = 5  # overnight batch: latency barely matters
    residency = 5 if model["deployment"] == "self-hosted" else 3
    multilingual = 4 if len(scenario["languages"]) > 1 and not model["open_weight"] else 3
    operability = 5 if model["deployment"] == "api" else 2
    return {"capability": capability, "latency": latency, "residency": residency,
            "multilingual": multilingual, "operability": operability}


scenario = SCENARIOS[SCENARIO_ID]
try:
    candidates = next(m.candidates for m in team_mappings() if m.scenario_id == SCENARIO_ID)
except NotImplementedError:
    candidates = [c for c in CATALOGUE if not CATALOGUE[c]["open_weight"]][:4]
    print("⏳ team_mappings() is a TODO: using the first four API models as candidates for now.")
pd.DataFrame({c: starter_ratings(CATALOGUE[c], scenario) for c in candidates}).T

In [ ]:
# (notebook) The two scoring functions.
import math


def cost_ratings(costs: dict[str, float]) -> dict[str, float]:
    """Map monthly costs to ratings: cheapest = 5.0, most expensive = 1.0, linear on a log10 scale.

    All costs equal → every model gets 5.0. Costs must be positive.
    """
    # TODO: take math.log10 of each cost, then scale linearly between the lowest (5.0) and highest (1.0)
    raise NotImplementedError("cost_ratings")


def weighted_score(ratings: dict[str, float], weights: dict[str, float]) -> float:
    """Sum of rating × weight. Weights must cover exactly the rated criteria and sum to 1.0."""
    # TODO: assert the criteria match and the weights sum to 1.0, then return the rounded weighted sum
    raise NotImplementedError("weighted_score")

In [ ]:
# (notebook) The decision matrix for your scenario, ranked.
def decision_matrix(candidates: list[str], scenario: dict, weights: dict[str, float]) -> pd.DataFrame:
    costs = {c: cost_or_none(CATALOGUE[c], scenario) for c in candidates}
    priced = {c: v for c, v in costs.items() if v}
    cost_rating = cost_ratings(priced) if priced else {}
    rows = {}
    for c in candidates:
        ratings = {**starter_ratings(CATALOGUE[c], scenario), "cost": cost_rating.get(c, 3.0)}  # unpriced → neutral 3
        rows[c] = {**ratings, "monthly $": costs[c], "score": weighted_score(ratings, weights)}
    return pd.DataFrame(rows).T.sort_values("score", ascending=False)


try:
    matrix = decision_matrix(candidates, scenario, WEIGHTS)
    display(matrix)
except NotImplementedError as exc:
    matrix = None
    print(f"⏳ {exc} is still a TODO")

In [ ]:
# (notebook) Sensitivity: does the winner survive a doubled cost weight, or a "self-hosted only" residency rule?
def renormalise(weights: dict[str, float]) -> dict[str, float]:
    total = sum(weights.values())
    return {k: v / total for k, v in weights.items()}


def leaders(frame: pd.DataFrame) -> list[str]:
    """Every candidate that shares the top score: more than one means a tie."""
    return frame.index[frame.score == frame.score.max()].tolist()


sensitivity_notes = []
if matrix is not None:
    top = leaders(matrix)
    if len(top) > 1:
        sensitivity_notes.append(f"Tie at {matrix.score.max()} between {', '.join(top)}: apply the tie-breaker you agreed before scoring.")
    heavier_cost = renormalise({**WEIGHTS, "cost": WEIGHTS["cost"] * 2})
    alt = leaders(decision_matrix(candidates, scenario, heavier_cost))
    sensitivity_notes.append(f"Doubling the cost weight: leader(s) {'unchanged' if alt == top else 'change to ' + ', '.join(alt)} ({', '.join(alt)}).")
    self_hosted = [c for c in candidates if CATALOGUE[c]["deployment"] == "self-hosted"]
    sensitivity_notes.append(
        f"A self-hosted-only residency rule leaves: {self_hosted or 'no candidate: add an open-weight model or a regional API'}."
    )
    print("\n".join(sensitivity_notes))

In [ ]:
# (notebook) Your team's decision and Responsible AI notes: edit the strings, then export the sheet.
TEAM = "Team name"
DECISION = {
    "tie_breaker": "TODO: the criterion you agreed BEFORE scoring, e.g. the existing contract or a second provider",
    "trade_off": "TODO: the one trade-off you accepted, in one sentence",
    "review_date": "TODO: when you will re-run this decision (model tables change within months)",
    "rai_bias": "TODO: how you will measure quality per language or region",
    "rai_transparency": "TODO: how users know the output is AI-generated and see its sources",
    "rai_governance": "TODO: who owns the model choice, prompts and evals; what is logged",
}


def markdown_table(frame: pd.DataFrame) -> str:
    columns = [str(c) for c in frame.columns]
    lines = ["| " + " | ".join([frame.index.name or ""] + columns) + " |", "| " + " | ".join(["---"] * (len(columns) + 1)) + " |"]
    for index, row in frame.iterrows():
        cells = ["—" if v is None or (isinstance(v, float) and math.isnan(v)) else str(v) for v in row.to_list()]
        lines.append("| " + " | ".join([str(index)] + cells) + " |")
    return "\n".join(lines)


def export_sheet(scenario_id: str) -> Path:
    """Fill templates/model-comparison-sheet.md for one scenario and write it to outputs/."""
    from datetime import date

    s = SCENARIOS[scenario_id]
    mapping = next(m for m in team_mappings() if m.scenario_id == scenario_id)
    top = leaders(matrix)
    profile = pd.DataFrame(
        {"value": {k: s[k] for k in ("team", "requests_per_day", "avg_input_tokens", "avg_output_tokens",
                                       "latency_slo_seconds", "residency", "data_sensitivity", "accuracy_bar")}}
    )
    candidates_frame = pd.DataFrame(
        {c: {"provider": CATALOGUE[c]["provider"], "tier": CATALOGUE[c]["tier"], "deployment": CATALOGUE[c]["deployment"],
             "context window": f"{CATALOGUE[c]['context_window']:,}", "monthly $": cost_or_none(CATALOGUE[c], s)}
         for c in mapping.candidates}
    ).T
    candidates_frame.index.name = "model"
    profile.index.name = "field"
    matrix.index.name = "model"
    fields = {
        "scenario_id": scenario_id, "scenario_title": s["title"], "team": TEAM, "date": date.today().isoformat(),
        "prices_as_of": CATALOGUE_DOC["prices_as_of"], "scenario_profile": markdown_table(profile),
        "agent_type": mapping.agent_type, "mapping_rationale": mapping.rationale, "model_tier": mapping.model_tier,
        "deployment": mapping.deployment, "candidates_table": markdown_table(candidates_frame),
        "matrix_table": markdown_table(matrix), "sensitivity": "\n".join(f"- {n}" for n in sensitivity_notes),
        "decision": " / ".join(top) + (" (tie: see the tie-breaker)" if len(top) > 1 else ""),
        "fallback": next((m for m in matrix.index if m not in top), "—"),
        "risks": "\n".join(f"- {r}" for r in mapping.risks), **DECISION,
    }
    sheet = (LAB_DIR / "templates" / "model-comparison-sheet.md").read_text(encoding="utf-8")
    for key, value in fields.items():
        sheet = sheet.replace("{{" + key + "}}", str(value))
    OUTPUT_DIR.mkdir(exist_ok=True)
    path = OUTPUT_DIR / f"model-comparison-{scenario_id}.md"
    path.write_text(sheet, encoding="utf-8")
    return path


sheet_path = None
if matrix is not None:
    try:
        sheet_path = export_sheet(SCENARIO_ID)
        print(f"Wrote {sheet_path.relative_to(LAB_DIR)}: open it, then edit the TODO lines with your team.")
    except NotImplementedError:
        print("⏳ Finish team_mappings() first: the sheet needs your mapping for this scenario.")

📝 **Write-up (3–5 sentences).** Which model did you choose for your scenario and why? Name the
criterion that decided it, the trade-off you accepted, and what evidence would make you change your mind.

> *Your answer here.*

In [ ]:
# (notebook) Check: the scoring functions meet their contracts and the sheet is complete.
def check_part4():
    ratings = cost_ratings({"a": 10.0, "b": 100.0, "c": 1_000.0})
    assert ratings == {"a": 5.0, "b": 3.0, "c": 1.0}, ratings
    assert cost_ratings({"a": 42.0, "b": 42.0}) == {"a": 5.0, "b": 5.0}
    assert weighted_score({"x": 4, "y": 2}, {"x": 0.75, "y": 0.25}) == 3.5
    try:
        weighted_score({"x": 4, "y": 2}, {"x": 0.5, "y": 0.4})
    except AssertionError:
        pass
    else:
        raise AssertionError("weighted_score must reject weights that do not sum to 1.0")
    assert math.isclose(sum(WEIGHTS.values()), 1.0), "your WEIGHTS must sum to 1.0"
    if sheet_path is None:
        raise NotImplementedError("export the sheet after team_mappings() is done")
    assert "{{" not in sheet_path.read_text(encoding="utf-8"), "every placeholder in the sheet must be filled"


check("Part 4 — scoring and sheet", check_part4)

---

**Done.** Answer the checkpoint questions in the lab guide. That completes the Module 1 labs:
see the README's clean-up section, then **Module 2 — Prompt Engineering, OpenAI & Claude SDKs,
LangChain & Single Agent**.